In [1]:
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
import pandas as pd
print("Module import success")

import sys
sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("Path append success")

import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))
print("success")

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund_gable")
env
print("Bootstrap environment success")

from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")
print("Utilities import success")

import matplotlib.pyplot as plt
import requests

from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")
raw_data=Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data")
processed_data=Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data")

c:\Users\edmund.gable\.conda\envs\satprocessing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Module import success
Path append success
⚠ Unrecognized preferred label 'edmund_gable'. Valid options: ['NECMAC04363461.local', 'nefscsatdata', 'guihyde', 'Mac.localdomain', 'gdavis', 'egable']
{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 10, 12, 10, 44, 770910),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RE

In [2]:
#Helper function that opens a DO dataset and drops any rows with missing DO values
def open_DO_dataset(file_path, DO_column_name):
    ds=xr.open_dataset(file_path)
    df=ds.to_dataframe()
    df=df.reset_index()
    df=df.dropna(subset=[DO_column_name])
    return df

In [36]:
# 3. Define the labeling function

def classify_DO(df, DO_column, battery_column=None):
    classified_df = df.copy()

    classified_df.loc[classified_df[DO_column] >= 3.9, 'DO_status'] = 'Low Oxygen'
    classified_df.loc[classified_df[DO_column] > 5.0, 'DO_status'] = 'Oxygenated' 
    classified_df.loc[classified_df[DO_column] < 3.9, 'DO_status'] = 'Mildly Hypoxic'
    classified_df.loc[classified_df[DO_column] < 2.41, 'DO_status'] = 'Hypoxia'
    classified_df.loc[classified_df[DO_column] < 0.2, 'DO_status'] = 'Anoxia'
    
    # 2. FILTER BATTERY (Only if a battery column name was provided)
    if battery_column is not None:
        classified_df = classified_df[classified_df[battery_column] >= 50]
        
    classified_df = classified_df.reset_index(drop=True)
    
    return classified_df

# 4. Run the function on your DataFrame
# You can now pass None explicitly, or just leave the third argument completely blank!
#Fishbot_hypoxia = classify_DO(dff, 'dissolved_oxygen')
# 4. Run the function on your DataFrame
#Fishbot_hypoxia = classify_DO(dff, 'dissolved_oxygen', None)

# 5. Print the columns to verify 'depth' is finally there
#print(Fishbot_hypoxia.columns.tolist())
#Fishbot_hypoxia
#Fishbot_hypoxia.to_excel(root / "data" / "Processed_Data" / "Fishbot_Hypoxia.xlsx", index=False)

def get_extent(df, time_column, lon_column, lat_column, depth_column=None):
    # 1. Convert to datetime and strip hours/minutes/seconds (normalizes to midnight)
    daily_time = pd.to_datetime(df[time_column]).dt.normalize()
    daily_time.name = 'time'
    # 2. Group by the new daily time
    lon_max = df.groupby(daily_time)[lon_column].max()
    lon_min=df.groupby(daily_time)[lon_column].max()
    lat_max = df.groupby(daily_time)[lat_column].max()
    lat_min = df.groupby(daily_time)[lat_column].min()
  
    # 3. Create the dataframe
    if depth_column is not None: 
      depth_max=df.groupby(daily_time)[depth_column].max()
      depth_min=df.groupby(daily_time)[depth_column].max()
      extent_df = pd.DataFrame({
        "lon_max": lon_max,
        "lon_min": lon_min,
        "lat_max": lat_max,
        "lat_min": lat_min,
        "depth_max": depth_max,
        "depth_min": depth_min
    })
    else:
      extent_df = pd.DataFrame({
        "lon_max": lon_max,
        "lon_min": lon_min,
        "lat_max": lat_max,
        "lat_min": lat_min})

    
    # 4. Reset index so 'time_column' becomes a regular column again instead of the index
    return extent_df.reset_index()

#def calculate_daily_averages(df, datetime_col, target_col, id_col):
    #df_working = df.copy()
    
    # 1. Normalize the time to midnight
    #df_working['daily_time'] = pd.to_datetime(df_working[datetime_col]).dt.normalize()
    
    # 2. Ensure the target column is numeric
    #df_working[target_col] = pd.to_numeric(df_working[target_col], errors='coerce')
    
    # *** THE FIX: Replace any absurdly high fill values with NaN ***
    # You can adjust this threshold (e.g., 50) based on what makes sense for your specific parameter!
    #df_working.loc[df_working[target_col] > 50, target_col] = np.nan
    
    # 3. Group by the daily_time (Pandas will automatically ignore the NaNs when calculating the mean)
    #daily_avg_df = df_working.groupby('daily_time')[target_col].mean().reset_index()
    
    # 4. Rename 'daily_time' back to the original time column name
    #avg_col_name = f'{target_col}_daily_avg'
    #daily_avg_df.rename(columns={'daily_time': datetime_col, target_col: avg_col_name}, inplace=True)
    
    #return daily_avg_df
def calculate_daily_averages(df, name_col, datetime_col, target_col):
    df_working = df.copy()
    
    # 1. Extract the date
    df_working['time'] = pd.to_datetime(df_working[datetime_col]).dt.date

    df_working['identifier'] = df_working[name_col]
    
    # 2. Ensure the raw aragonite column is numeric
    df_working[target_col] = pd.to_numeric(df_working[target_col], errors='coerce')
    
    # 3. Group by vessel and time, then calculate the mean of the raw numbers
    daily_avg_df = df_working.groupby(['identifier', 'time'])[target_col].mean().reset_index()
    
    # 4. Rename the column so we know it is a daily average
    avg_col_name = f'{target_col}_daily_avg'
    daily_avg_df.rename(columns={target_col: avg_col_name}, inplace=True)
    
    return daily_avg_df
def combine_df(df1, df2, join_key1):
    # Make copies so we don't accidentally alter the original dataframes
    df1_working = df1.copy()
    df2_working = df2.copy()
    
    # Force the join key in both dataframes to be a standard Pandas datetime object
    df1_working[join_key1] = pd.to_datetime(df1_working[join_key1])
    df2_working[join_key1] = pd.to_datetime(df2_working[join_key1])
    
    # Now that the data types perfectly match, perform the merge
    combined_df = pd.merge(df1_working, df2_working, on=join_key1)
    
    return combined_df
def filter_df(df, empty_col):
   return df.dropna(subset=[empty_col]).reset_index(drop=True)
   


In [4]:
#1. Open the dataset and check the column headers
ds = xr.open_dataset(root / raw_data / "eMOLT" / "eMOLT_2000_2026.nc")
dfeM = ds.to_dataframe().reset_index() 
print(dfeM.columns.to_list())
print(dfeM)

#2. calculate daily averages 
dfeMd = calculate_daily_averages(dfeM, 'tow_id', 'time', 'DO')

#3. classify the daily averages
dfeMdc = classify_DO(dfeMd, 'DO_daily_avg')

#  Force time to midnight so it matches exactly ***
dfeMdc['time'] = pd.to_datetime(dfeMdc['time']).dt.normalize()

#4. get spatial extent
dfeMe = get_extent(dfeM, 'time', 'longitude', 'latitude') 

#5. Reset the index and ensure the column is named 'time' 
# Use rename just in case the groupby index lost its 'time' name
#dfeMe = dfeMe.reset_index().rename(columns={'index': 'time'})

#5. combine spatial extent with classified daily averages
df_final = combine_df(dfeMdc, dfeMe, 'time')
print(f"Distinct dates after merge: {df_final['time'].nunique()}")

#6. export dataset
export_path = root / "data" / "Processed_Data" / "eMOLT_DO_extent_average_2023_2026.csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d', index=False)
#6. export dataset
export_path = root / "data" / "Processed_Data" / "eMOLT_DO_extent_average_2023_2026.csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)

['row', 'tow_id', 'time', 'latitude', 'longitude', 'temperature', 'DO', 'DO_percentage', 'water_detect_perc', 'sensor_type']
           row   tow_id                time   latitude  longitude  \
0            0   4295.0 2023-07-11 17:35:58  41.662701 -70.362198   
1            1   4295.0 2023-07-11 17:50:58  41.662701 -70.362198   
2            2   4295.0 2023-07-11 18:05:58  41.662701 -70.362198   
3            3   4295.0 2023-07-11 18:20:58  41.662701 -70.362198   
4            4   4295.0 2023-07-11 18:35:58  41.662701 -70.362198   
...        ...      ...                 ...        ...        ...   
981802  981802  57701.0 2026-05-24 22:53:49  43.131500 -69.461800   
981803  981803  57701.0 2026-05-24 23:08:49  43.131500 -69.461800   
981804  981804  57701.0 2026-05-24 23:23:49  43.131500 -69.461800   
981805  981805  57701.0 2026-05-24 23:38:49  43.131500 -69.461800   
981806  981806  57701.0 2026-05-24 23:53:49  43.131500 -69.461800   

        temperature     DO  DO_percentage  wat

In [ ]:
dfeM

In [ ]:
#1. Open the dataset and check the column headers
ds = xr.open_dataset(root / raw_data / "eMOLT" / "eMOLT_2000_2026.nc")
dfeM = ds.to_dataframe().reset_index() 
print(dfeM.columns.to_list())

#2. calculate daily averages 
dfeMd = calculate_daily_averages(dfeM, 'time', 'DO')

#3. classify the daily averages
dfeMdc = classify_DO(dfeMd, 'DO_daily_avg')

#  Force time to midnight so it matches exactly ***
dfeMdc['time'] = pd.to_datetime(dfeMdc['time']).dt.normalize()

#4. get spatial extent
dfeMe = get_extent(dfeM, 'time', 'longitude', 'latitude') 

#5. Reset the index and ensure the column is named 'time' 
# Use rename just in case the groupby index lost its 'time' name
#dfeMe = dfeMe.reset_index().rename(columns={'index': 'time'})

#5. combine spatial extent with classified daily averages
df_final = combine_df(dfeMdc, dfeMe, 'time')
print(f"Distinct dates after merge: {df_final['time'].nunique()}")

#6. export dataset
export_path = root / "data" / "Processed_Data" / "eMOLT_DO_extent_average_2023_2026.csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d', index=False)
#6. export dataset
export_path = root / "data" / "Processed_Data" / "eMOLT_DO_extent_average_2023_2026.csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)

,time,date,DO_daily_avg
0,2023-07-11 17:32:44,2023-07-11,7.63
1,2023-07-11 17:35:44,2023-07-11,7.30
2,2023-07-11 17:35:58,2023-07-11,7.36
3,2023-07-11 17:36:12,2023-07-11,7.42
4,2023-07-11 17:39:48,2023-07-11,7.29
...,...,...,...
958133,2026-05-24 22:53:49,2026-05-24,10.65
958134,2026-05-24 23:08:49,2026-05-24,10.64
958135,2026-05-24 23:23:49,2026-05-24,10.64
958136,2026-05-24 23:38:49,2026-05-24,10.64


In [13]:
#1. Open the dataset and check the column headers
ds = xr.open_dataset(root / raw_data / "Hydrographic_Monitoring" / "errdap_search.nc")
dfeM = ds.to_dataframe().reset_index() 
print(dfeM.columns.to_list())

#2. calculate daily averages 
dfeMd = calculate_daily_averages(dfeM, 'cruise_id', 'UTC_DATETIME', 'dissolved_oxygen')

print(dfeMd.columns.to_list())

#3. classify the daily averages
dfeMdc = classify_DO(dfeMd, 'dissolved_oxygen_daily_avg')
print(dfeMdc.columns.to_list())

#  Force time to midnight so it matches exactly ***
dfeMdc['time'] = pd.to_datetime(dfeMdc['time']).dt.normalize()

#4. get spatial extent
dfeMe = get_extent(dfeM, 'UTC_DATETIME', 'longitude', 'latitude') 

#5. Reset the index and ensure the column is named 'time' 
# Use rename just in case the groupby index lost its 'time' name
dfeMe = dfeMe.reset_index().rename(columns={'UTC_DATETIME': 'time'})
dfeMdc['time'] = pd.to_datetime(dfeMdc['time']).astype('datetime64[ns]')
dfeMe['time'] = pd.to_datetime(dfeMe['time']).astype('datetime64[ns]')

# normalize again just to be safe
dfeMdc['time'] = dfeMdc['time'].dt.normalize()
dfeMe['time'] = dfeMe['time'].dt.normalize()

#5. combine spatial extent with classified daily averages
df_final = combine_df(dfeMdc, dfeMe, 'time')
print(f"Distinct dates after merge: {df_final['time'].nunique()}")

#6. export dataset
export_path = root / "data" / "Processed_Data" / "Hydrographic_Monitoring.csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)

['row', 'UTC_DATETIME', 'latitude', 'longitude', 'depth', 'dissolved_oxygen', 'cruise_id', 'purpose_code', 'bottom_depth', 'GEAR_TYPE']
['identifier', 'time', 'dissolved_oxygen_daily_avg']
['identifier', 'time', 'dissolved_oxygen_daily_avg', 'DO_status']
Distinct dates after merge: 1464


In [ ]:
df_final

datetime64[ns]
datetime64[ns]
0


In [27]:
#1. Open the dataset and check the column headers
ds = xr.open_dataset( root / raw_data / "Fishbot" / "ondeckdata_erddap6.nc")
df = ds.to_dataframe().reset_index() 
print(df.columns.to_list())

#2. calculate daily averages 
dfd = calculate_daily_averages(df, 'data_provider', 'time', 'dissolved_oxygen')

print(dfd.columns.to_list())

#3. classify the daily averages
dfdc = classify_DO(dfd, 'dissolved_oxygen_daily_avg')
print(dfdc.columns.to_list())

#  Force time to midnight so it matches exactly ***
dfdc['time'] = pd.to_datetime(dfdc['time']).dt.normalize()

#4. get spatial extent
dfe = get_extent(df, 'time', 'longitude', 'latitude') 

#5. Reset the index and ensure the column is named 'time' 
# Use rename just in case the groupby index lost its 'time' name
dfe = dfe.reset_index().rename(columns={'time': 'time'})
dfdc['time'] = pd.to_datetime(dfdc['time']).astype('datetime64[ns]')
dfe['time'] = pd.to_datetime(dfe['time']).astype('datetime64[ns]')

# normalize again just to be safe
dfdc['time'] = dfdc['time'].dt.normalize()
dfe['time'] = dfe['time'].dt.normalize()

#5. combine spatial extent with classified daily averages
df_final = combine_df(dfdc, dfe, 'time')
print(f"Distinct dates after merge: {df_final['time'].nunique()}")

#6. remove any rows where DO is nan
df_final=filter_df(df_final,'dissolved_oxygen_daily_avg')

#6. export dataset
export_path = root / "data" / "Processed_Data" / "ondeckdata_extent_daily.csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)

['row', 'time', 'temperature', 'dissolved_oxygen', 'dissolved_oxygen_min', 'dissolved_oxygen_max', 'dissolved_oxygen_std', 'salinity', 'data_provider', 'grid_id', 'stat_area', 'fishery_dependent', 'depth', 'latitude', 'longitude']
['identifier', 'time', 'dissolved_oxygen_daily_avg']
['identifier', 'time', 'dissolved_oxygen_daily_avg', 'DO_status']
Distinct dates after merge: 8213


In [45]:
#1. Open the dataset and check the column headers
df = pd.read_csv( root / raw_data / "BCO_DMO" / "966896_v1_ooi_pioneer_nes_ctd_and_water_sampling_data.csv")
#df = ds.to_dataframe().reset_index() 
print(df.columns.to_list())
df=df[df['CTD_Depth']<= (df['Bottom_Depth_at_Start_Position']-20)].reset_index(drop=True)

#2. calculate daily averages 
dfd = calculate_daily_averages(df, 'Target_Asset', 'Start_Time', 'CTD_Oxygen')
# convert mL/L to mg/L
dfd['CTD_Oxygen_daily_avg'] = (dfd['CTD_Oxygen_daily_avg'] * 1.42905)


print(dfd.columns.to_list())

#3. classify the daily averages
dfdc = classify_DO(dfd, 'CTD_Oxygen_daily_avg')
print(dfdc.columns.to_list())

#  Force time to midnight so it matches exactly ***
dfdc['time'] = pd.to_datetime(dfdc['time']).dt.normalize()

#4. get spatial extent
dfe = get_extent(df, 'Start_Time', 'Start_Longitude', 'Start_Latitude') 

#5. Reset the index and ensure the column is named 'time' 
# Use rename just in case the groupby index lost its 'time' name
dfe = dfe.reset_index().rename(columns={'time': 'time'})
dfdc['time'] = pd.to_datetime(dfdc['time'], utc=True).dt.tz_localize(None)
dfe['time'] = pd.to_datetime(dfe['time'], utc=True).dt.tz_localize(None)

dfdc['time'] = dfdc['time'].dt.normalize()
dfe['time'] = dfe['time'].dt.normalize()

#5. combine spatial extent with classified daily averages
df_final = combine_df(dfdc, dfe, 'time')
print(f"Distinct dates after merge: {df_final['time'].nunique()}")

#6. remove any rows where DO is nan
df_final=filter_df(df_final,'CTD_Oxygen_daily_avg')

#6. export dataset
export_path = root / "data" / "Processed_Data" / "OOI_Pioneer_extent_daily_2014_2022).csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)

['Cruise', 'Station', 'Target_Asset', 'Start_Latitude', 'Start_Longitude', 'Start_Time', 'Cast', 'Cast_Flag', 'Bottom_Depth_at_Start_Position', 'CTD_File', 'CTD_File_Flag', 'Niskin_Bottle_Position', 'Niskin_Flag', 'CTD_Bottle_Closure_Time', 'CTD_Pressure', 'CTD_Pressure_Flag', 'CTD_Depth', 'CTD_Latitude', 'CTD_Longitude', 'CTD_Temperature_1', 'CTD_Temperature_1_Flag', 'CTD_Temperature_2', 'CTD_Temperature_2_Flag', 'CTD_Conductivity_1', 'CTD_Conductivity_1_Flag', 'CTD_Conductivity_2', 'CTD_Conductivity_2_Flag', 'CTD_Salinity_1', 'CTD_Salinity_2', 'CTD_Oxygen', 'CTD_Oxygen_Flag', 'CTD_Oxygen_Saturation', 'CTD_Fluorescence', 'CTD_Fluorescence_Flag', 'CTD_Beam_Attenuation', 'CTD_Beam_Transmission', 'CTD_Transmissometer_Flag', 'CTD_pH', 'CTD_pH_Flag', 'Discrete_Oxygen', 'Discrete_Oxygen_Flag', 'Discrete_Oxygen_Replicate_Flag', 'Discrete_Chlorophyll', 'Discrete_Phaeopigment', 'Discrete_Fo_Fa_Ratio', 'Discrete_Fluorescence_Flag', 'Discrete_Fluorescence_Replicate_Flag', 'Discrete_Phosphate', '

In [37]:
def combine_df(df1, df2, merge_col):
    return pd.merge(df1, df2, on=merge_col)
print(dfeMdc.shape)
print(dfeMe.shape)

print(dfeMdc['time'].nunique())
print(dfeMe['time'].nunique())

common = set(dfeMdc['time']) & set(dfeMe['time'])
print(len(common))
test = pd.merge(
    dfeMdc,
    dfeMe,
    on='time',
    how='inner'
)

print(test.shape)
print(test.head())



(1597, 4)
(1464, 6)
1464
1
0
(0, 9)
Empty DataFrame
Columns: [identifier, time, dissolved_oxygen_daily_avg, DO_status, UTC_DATETIME, lon_max, lon_min, lat_max, lat_min]
Index: []
